In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from vit import ViT
import time  # Добавили модуль времени

def main():


    model = ViT(
        img_size=224,
        patch_size=16,
        num_classes=10,
        in_channels=3,
        dim=384,
        depth=7,
        heads=6
    )

    batch_size = 32

    # Нормализация ImageNet
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    # Аугментации для train 
    transform_train = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # ← ЗАПЯТАЯ!
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        normalize
    ])

    # Для test/val
    transform_test = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        normalize
    ])

    train_dataset = torchvision.datasets.Imagenette(
        root='./data',
        split='train',
        size='320px',
        download=False,
        transform=transform_train
    )

    test_dataset = torchvision.datasets.Imagenette(
        root='./data',
        split='val',
        size='320px',
        download=False,
        transform=transform_test
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)


    def train_epoch(model, loader, optimizer, criterion, device):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        return total_loss / len(loader), correct / total

    def test_epoch(model, loader, criterion, device):
        model.eval()
        total_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                
                total_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        return total_loss / len(loader), correct / total

    # Настройки цикла
    epochs = 50
    test_accs = []
    epoch_times = []  # Список для хранения времени каждой эпохи
    best_test_loss = float('inf')

    # === EARLY STOPPING ===
    patience = 7
    epochs_no_improve = 0

    log_file = open("training_log.txt", "w", encoding="utf-8")

    print(f"Starting training on {device}...")

    for epoch in range(epochs):
        start_time = time.time()  # Засекаем время начала эпохи
        
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        test_loss, test_acc = test_epoch(model, test_loader, criterion, device)
        scheduler.step()
        
        end_time = time.time()  # Засекаем время окончания
        epoch_duration = end_time - start_time
        epoch_times.append(epoch_duration)
        
        test_accs.append(test_acc)

        # Подготовка строки лога
        log_str = (f"Epoch {epoch + 1}/{epochs} | "
                   f"Time: {epoch_duration:.2f}s | "
                   f"Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f} | "
                   f"Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}\n")
        
        print(log_str, end="")
        log_file.write(log_str)
        log_file.flush()

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), 'best_vit_model.pth')
            print(f"---> New best model saved! Loss: {test_loss:.4f}")
        else:
            epochs_no_improve += 1
            print(f"No improvement: {epochs_no_improve}/{patience}")

        # === EARLY STOPPING ===
        if epochs_no_improve >= patience:
            print(f"\n Early stopping at epoch {epoch+1}")
            break

        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), f'vit_epoch_{epoch+1}.pth')

    # Итоговая статистика
    avg_time = sum(epoch_times) / len(epoch_times)
    total_time = sum(epoch_times)
    
    summary_str = (f"\n{'='*30}\n"
                   f"Training Complete!\n"
                   f"Total Time: {total_time:.2f}s ({total_time/60:.2f} min)\n"
                   f"Average Time per Epoch: {avg_time:.2f}s\n"
                   f"Best Test Accuracy: {max(test_accs):.4f}\n"
                   f"{'='*30}")
    
    print(summary_str)
    log_file.write(summary_str + "\n")
    log_file.close()

if __name__ == "__main__":
    main()


Starting training on cuda...
Epoch 1/50 | Time: 28.71s | Train Acc: 0.2568, Test Acc: 0.3901 | Train Loss: 2.0875, Test Loss: 1.8057
---> New best model saved! Loss: 1.8057
Epoch 2/50 | Time: 31.42s | Train Acc: 0.4071, Test Acc: 0.4831 | Train Loss: 1.7344, Test Loss: 1.5123
---> New best model saved! Loss: 1.5123
Epoch 3/50 | Time: 33.75s | Train Acc: 0.4593, Test Acc: 0.5671 | Train Loss: 1.5776, Test Loss: 1.3256
---> New best model saved! Loss: 1.3256
Epoch 4/50 | Time: 34.67s | Train Acc: 0.5043, Test Acc: 0.5577 | Train Loss: 1.4631, Test Loss: 1.3033
---> New best model saved! Loss: 1.3033
Epoch 5/50 | Time: 35.39s | Train Acc: 0.5199, Test Acc: 0.5676 | Train Loss: 1.4242, Test Loss: 1.2886
---> New best model saved! Loss: 1.2886
Epoch 6/50 | Time: 35.66s | Train Acc: 0.5424, Test Acc: 0.5783 | Train Loss: 1.3571, Test Loss: 1.2820
---> New best model saved! Loss: 1.2820
Epoch 7/50 | Time: 35.81s | Train Acc: 0.5615, Test Acc: 0.6456 | Train Loss: 1.3115, Test Loss: 1.0967
---

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from vit import ViT
import time  # Добавили модуль времени

def main():


    model = ViT(
        img_size=224,
        patch_size=16,
        num_classes=10,
        in_channels=3,
        dim=512,
        depth=10,
        heads=8
    )

    batch_size = 32

    # Нормализация ImageNet
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    # Аугментации для train 
    transform_train = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # ← ЗАПЯТАЯ!
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        normalize
    ])

    # Для test/val
    transform_test = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        normalize
    ])

    train_dataset = torchvision.datasets.Imagenette(
        root='./data',
        split='train',
        size='320px',
        download=False,
        transform=transform_train
    )

    test_dataset = torchvision.datasets.Imagenette(
        root='./data',
        split='val',
        size='320px',
        download=False,
        transform=transform_test
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)


    def train_epoch(model, loader, optimizer, criterion, device):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        return total_loss / len(loader), correct / total

    def test_epoch(model, loader, criterion, device):
        model.eval()
        total_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                
                total_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        return total_loss / len(loader), correct / total

    # Настройки цикла
    epochs = 50
    test_accs = []
    epoch_times = []  # Список для хранения времени каждой эпохи
    best_test_loss = float('inf')

    # === EARLY STOPPING ===
    patience = 7
    epochs_no_improve = 0

    log_file = open("training_log.txt", "w", encoding="utf-8")

    print(f"Starting training on {device}...")

    for epoch in range(epochs):
        start_time = time.time()  # Засекаем время начала эпохи
        
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        test_loss, test_acc = test_epoch(model, test_loader, criterion, device)
        scheduler.step()
        
        end_time = time.time()  # Засекаем время окончания
        epoch_duration = end_time - start_time
        epoch_times.append(epoch_duration)
        
        test_accs.append(test_acc)

        # Подготовка строки лога
        log_str = (f"Epoch {epoch + 1}/{epochs} | "
                   f"Time: {epoch_duration:.2f}s | "
                   f"Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f} | "
                   f"Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}\n")
        
        print(log_str, end="")
        log_file.write(log_str)
        log_file.flush()

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), 'best_vit_model.pth')
            print(f"---> New best model saved! Loss: {test_loss:.4f}")
        else:
            epochs_no_improve += 1
            print(f"No improvement: {epochs_no_improve}/{patience}")

        # === EARLY STOPPING ===
        if epochs_no_improve >= patience:
            print(f"\n Early stopping at epoch {epoch+1}")
            break

        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), f'vit_epoch_{epoch+1}.pth')

    # Итоговая статистика
    avg_time = sum(epoch_times) / len(epoch_times)
    total_time = sum(epoch_times)
    
    summary_str = (f"\n{'='*30}\n"
                   f"Training Complete!\n"
                   f"Total Time: {total_time:.2f}s ({total_time/60:.2f} min)\n"
                   f"Average Time per Epoch: {avg_time:.2f}s\n"
                   f"Best Test Accuracy: {max(test_accs):.4f}\n"
                   f"{'='*30}")
    
    print(summary_str)
    log_file.write(summary_str + "\n")
    log_file.close()

if __name__ == "__main__":
    main()


Starting training on cuda...
Epoch 1/50 | Time: 75.93s | Train Acc: 0.2599, Test Acc: 0.4313 | Train Loss: 2.0875, Test Loss: 1.6922
---> New best model saved! Loss: 1.6922
Epoch 2/50 | Time: 78.25s | Train Acc: 0.4367, Test Acc: 0.5218 | Train Loss: 1.6624, Test Loss: 1.4340
---> New best model saved! Loss: 1.4340
Epoch 3/50 | Time: 80.58s | Train Acc: 0.4960, Test Acc: 0.5852 | Train Loss: 1.4948, Test Loss: 1.2385
---> New best model saved! Loss: 1.2385
Epoch 4/50 | Time: 79.40s | Train Acc: 0.5361, Test Acc: 0.5885 | Train Loss: 1.3827, Test Loss: 1.2309
---> New best model saved! Loss: 1.2309
Epoch 5/50 | Time: 79.58s | Train Acc: 0.5605, Test Acc: 0.6135 | Train Loss: 1.3149, Test Loss: 1.1509
---> New best model saved! Loss: 1.1509
Epoch 6/50 | Time: 77.93s | Train Acc: 0.5887, Test Acc: 0.6601 | Train Loss: 1.2320, Test Loss: 1.0391
---> New best model saved! Loss: 1.0391
Epoch 7/50 | Time: 80.04s | Train Acc: 0.6069, Test Acc: 0.6637 | Train Loss: 1.1943, Test Loss: 1.0238
---